# 00 — Configuração dos municípios

Este notebook prepara uma execução local. Há duas formas de informar os municípios:

1. criar `municipios.csv` na raiz do repositório, com uma coluna `id_municipio`;
2. preencher `MUNICIPIOS_INLINE` abaixo.

Os códigos devem ser códigos IBGE municipais de 7 dígitos. Nenhum município é pré-selecionado.


In [ ]:
from pathlib import Path
import os, json, math
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "dados"
OUT_DIR = DATA_DIR / "processado"
CONTROL_DIR = DATA_DIR / "controle"
for p in [DATA_DIR, OUT_DIR, CONTROL_DIR]: p.mkdir(parents=True, exist_ok=True)

ARQUIVO_MUNICIPIOS = ROOT / "municipios.csv"
MUNICIPIOS_INLINE = [
    # "3516408",  # Franco da Rocha
    # "3525904",  # Jundiaí
]
LOTE_TAMANHO = 5

if ARQUIVO_MUNICIPIOS.exists():
    mun = pd.read_csv(ARQUIVO_MUNICIPIOS, dtype=str)
    if "id_municipio" not in mun.columns:
        raise ValueError("municipios.csv deve conter a coluna id_municipio")
    ids = mun["id_municipio"].astype(str).str.strip().dropna().tolist()
else:
    ids = [str(x).strip() for x in MUNICIPIOS_INLINE if str(x).strip()]

ids = list(dict.fromkeys(ids))
if not ids:
    raise ValueError("Informe municípios em municipios.csv ou em MUNICIPIOS_INLINE.")
if any(len(x) != 7 or not x.isdigit() for x in ids):
    raise ValueError("Todos os códigos devem ser códigos IBGE municipais de 7 dígitos.")

LOTES = [ids[i:i+LOTE_TAMANHO] for i in range(0, len(ids), LOTE_TAMANHO)]
print(f"Municípios: {len(ids)} | lotes: {len(LOTES)} | tamanho máximo: {LOTE_TAMANHO}")

config = {"municipios": ids, "lote_tamanho": LOTE_TAMANHO}
(CONTROL_DIR / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
print("Configuração salva em", CONTROL_DIR / "config.json")
